# Automotive Service Manual RAG Pipeline — Data Cleaning & EDA

This notebook extracts text from GM service manual PDFs (Avalanche, Escalade, Suburban, Tahoe, Yukon 2007–2013), performs exploratory data analysis on the raw text, and builds a cleaning pipeline to remove OCR artifacts, special characters, and repeated metadata/boilerplate lines before the text is chunked and embedded for a RAG system.

## 1. Load & Extract Text from PDFs

Open the source PDFs with PyMuPDF (`fitz`) and extract raw text page by page into a single list of documents.

In [ ]:
import pymupdf

pdf1 = fitz.open(r"/content/كتيب PDF صيانه جمس - تاهو - افلنش - اسكاليد 2007-2013 HECTRsa النسخه 2.pdf")
pdf2 = fitz.open(r"/content/كتيب PDF صيانه جمس - تاهو - افلنش - اسكاليد 2007-2013 HECTRsa.pdf")

documents = []

for doc in [pdf1, pdf2]:

    print(f"Reading {doc.name}")

    for page in doc:
        text = page.get_text()

        if text.strip():
            documents.append(text)

    doc.close()

print(f"Total pages extracted: {len(documents)}")

## 2. Exploratory Data Analysis (Raw Text)

Inspect basic statistics of the raw corpus before any cleaning is applied.

### 2.1 Word Count Overview

In [ ]:
from collections import Counter

all_text = " ".join(documents)

words = all_text.split()

print(f"Total words: {len(words):,}")
print(f"Unique words: {len(set(words)):,}")

### 2.2 Most Frequent Lines

Identify repeated boilerplate/metadata lines (e.g. publisher notices, watermark text) that appear across many pages and should be removed.

In [ ]:
from collections import Counter

lines = []

for page in documents:
    for line in page.splitlines():
        line = line.strip()

        if line:
            lines.append(line)

counter = Counter(lines)

for line, count in counter.most_common(50):
    print(count, line)

### 2.3 Special Character Frequency

Count non-alphanumeric, non-whitespace characters to identify OCR glyphs, encoding artifacts, and symbols that need normalization or removal.

In [ ]:
from collections import Counter

special_chars = Counter()

for page in documents:
    for ch in page:
        if not ch.isalnum() and not ch.isspace():
            special_chars[ch] += 1

special_chars.most_common(50)

## 3. Cleaning Functions

Three modular cleaning functions are defined, each responsible for a single concern, so they can be composed and tested independently.

### 3.1 General Text Normalization

Standardize line endings, collapse redundant whitespace/blank lines, and trim leading/trailing spaces.

In [ ]:
import re

def general_cleaning(text):
    # توحيد نهاية الأسطر
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # حذف المسافات الزائدة
    text = re.sub(r"[ \t]+", " ", text)

    # حذف الأسطر الفارغة
    text = re.sub(r"\n{2,}", "\n", text)

    # إزالة المسافات في بداية ونهاية الملف
    text = text.strip()

    return text

### 3.2 Symbol Normalization

Replace OCR glyph artifacts (private-use Unicode codepoints) and typographic symbols (smart quotes, dashes, bullets) with clean ASCII equivalents.

In [ ]:
import re

def clean_symbols(text):
    replacements = {
        "\uf06c": "",
        "\uf0a1": "",
        "\uf0a7": "",
        "•": "",
        "®": "",
        "™": "",
        "©": "",
        "✍": "",
        "¨": "",

        "—": "-",
        "–": "-",
        "−": "-",

        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text

### 3.3 Metadata & Boilerplate Removal

Strip repeated publisher notices and watermark lines (e.g. "Courtesy of GENERAL MOTORS CORP.", "cardiagn.com") identified during EDA.

In [ ]:
import re

def remove_metadata(text):
    cleaned_lines = []

    for line in text.splitlines():
        if re.search(r"Courtesy of GENERAL MOTORS CORP\.?", line, flags=re.IGNORECASE):
            continue

        if re.search(r"cardiagn\.com", line, flags=re.IGNORECASE):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

## 4. Apply the Cleaning Pipeline

Run each page through the three cleaning functions in sequence and collect the results in `cleaned_documents`.

In [ ]:
cleaned_documents = []

for page in documents:
    page = general_cleaning(page)
    page = clean_symbols(page)
    page = remove_metadata(page)

    cleaned_documents.append(page)

## 5. Validate the Cleaning Results

Re-run the same diagnostics as Section 2, but on `cleaned_documents`, to confirm metadata lines and special characters have been removed.

### 5.1 Most Frequent Lines (Post-Cleaning)

In [ ]:
from collections import Counter

lines = []

for page in cleaned_documents:
    for line in page.splitlines():
        line = line.strip()

        if line:
            lines.append(line)

counter = Counter(lines)

for line, count in counter.most_common(50):
    print(count, line)

### 5.2 Special Character Frequency (Post-Cleaning)

In [ ]:
from collections import Counter

special_chars = Counter()

for page in cleaned_documents:
    for ch in page:
        if not ch.isalnum() and not ch.isspace():
            special_chars[ch] += 1

special_chars.most_common(50)

### 5.3 Spot-Check a Single Page

Locate pages that originally contained metadata, then print one before/after example to visually confirm the fix.

In [ ]:
for i, page in enumerate(documents):
    if "Courtesy" in page or "cardiagn.com" in page:
        print(i)

In [ ]:
page = documents[0]

page = general_cleaning(page)
page = clean_symbols(page)
page = remove_metadata(page)

print(page)

In [ ]:
print(len(documents))
print(documents[0][:500])